#1.Import Libraries

In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

#2. Load and Preprocess IMDB Dataset

In [2]:
# Vocabulary size
vocab_size = 10000

# Load dataset
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

# Pad sequences
max_length = 200
X_train = pad_sequences(X_train, maxlen=max_length, padding='post')
X_test = pad_sequences(X_test, maxlen=max_length, padding='post')

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


#3. Build Bi-directional LSTM Model

In [3]:
model = Sequential()

# Embedding layer
model.add(Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length))

# Bi-directional LSTM
model.add(Bidirectional(LSTM(64, return_sequences=False)))

# Dropout for regularization
model.add(Dropout(0.5))

# Fully connected layer
model.add(Dense(64, activation='relu'))

# Output layer
model.add(Dense(1, activation='sigmoid'))

# Compile model
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


#4. Train the Model

In [4]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 100s 312ms/step - accuracy: 0.7476 - loss: 0.5104 - val_accuracy: 0.8484 - val_loss: 0.3820
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 92s 293ms/step - accuracy: 0.8911 - loss: 0.2849 - val_accuracy: 0.8694 - val_loss: 0.3150
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 142s 295ms/step - accuracy: 0.9313 - loss: 0.1920 - val_accuracy: 0.8752 - val_loss: 0.3381
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 94s 300ms/step - accuracy: 0.9514 - loss: 0.1401 - val_accuracy: 0.8608 - val_loss: 0.3807
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 146s 314ms/step - accuracy: 0.9626 - loss: 0.1105 - val_accuracy: 0.8260 - val_loss: 0.5235


#5. Evaluate Model

In [5]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy:.4f}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 30s 38ms/step - accuracy: 0.8148 - loss: 0.5438
Test Accuracy: 0.8148


#6. Make Predictions

In [6]:
def predict_review(text_sequence):
    padded = pad_sequences([text_sequence], maxlen=max_length, padding='post')
    prediction = model.predict(padded)[0][0]
    return "Positive" if prediction > 0.5 else "Negative"